# Imports

In [359]:
import os
import numpy as np
from typing import Callable, List, Tuple, Any
from itertools import combinations

# CSV Reader

In [360]:
class CSVReader:
    @classmethod
    def __extract(cls, line: str):
        elements = line.split(",")
        for i in range(len(elements)):
            element_i: str = elements[i].strip().lower()
            try:
                elements[i] = float(element_i)
            except ValueError:
                elements[i] = element_i
        return elements
    
    @classmethod
    def fromFile(cls, filename: str) -> "CSVReader":
        values = []
        with open(filename) as file:
            columns = tuple(cls.__extract(file.readline()))
            for index, line in enumerate(file):
                row = cls.__extract(line)
                assert len(row) == len(columns), (
                    f"Row {index} has {len(row)} values but expected {len(columns)}"
                )
                values.append(row)
        return cls(columns, values)


    def __init__(self, columns: tuple, values: list):
        self.columns = columns
        self.values = values
        
    def __str__(self):
        s = f"{self.columns}"
        for line in self.values:
            s += f"\n{line}"
        return s
    
    def __len__(self):
        return len(self.values)
    
    def getColumns(self) -> tuple:
        return self.columns
    
    def getValues(self) -> list:
        return self.values

    def map(self, f: Callable[[List], Any], in_place: bool = False) -> "CSVReader":
        if in_place:
            reader = self
            values = self.getValues()
        else:
            reader = self.copy()
            values = reader.getValues()

        for i in range(len(values)):
            values[i] = f(values[i])

        return reader

    def reduce(self, f: Callable[[List, Any], Any], init_value: Any) -> Any:
        value: Any = init_value
        for line in self.values:
            value = f(line, value)
        return value
    
    def union(self, other: "CSVReader") -> "CSVReader":
        assert len(self) == len(other), f"Readers have different sizes {len(self)} vs {len(other)}"
        columns = self.getColumns() + other.getColumns()
        values0 = self.getValues()
        values1 = other.getValues()
        size = len(self)
        newValues = [None] * size
        for i in range(size):
            l0, l1 = values0[i], values1[i]
            newValues[i] = [None] * (len(l0) + len(l1))
            k = 0
            for j in range(len(l0)):
                newValues[i][k] = l0[j]
                k += 1
            for j in range(len(l1)):
                newValues[i][k] = l1[j]
                k += 1
        return CSVReader(columns, newValues)

    def copy(self) -> "CSVReader":
        newValue = [None] * len(self.values)
        for j in range(len(self.values)):
            line = self.values[j]
            newValue[j] = [None] * len(line)
            for i in range(len(line)):
                newValue[j][i] = line[i]        
        return CSVReader(self.columns, newValue)
    

# Part 1

In [361]:
def k_anonymity(dataset: CSVReader, indexes: list) -> int:
    def f(attributes: list, accumulator: dict) -> dict:
        group = [None] * len(indexes)
        k = 0
        for i in indexes:
            group[k] = attributes[i]
            k += 1
        h = hash(f"{group}")
        if accumulator.get(h, None) == None:
            accumulator[h] = 1
        else:
            accumulator[h] += 1
        return accumulator
    results: dict = dataset.reduce(f, {})
    return min(results.values())

def distance(dataset: CSVReader, protected_dataset: CSVReader) -> float:
    assert dataset.getColumns() == protected_dataset.getColumns(), \
        f"Attributes mismatch {dataset.getColumns()} vs {protected_dataset.getColumns()}"
    assert len(dataset) == len(protected_dataset), \
        f"Sizes mismatch {len(dataset)} vs {len(protected_dataset)}"
    size = len(dataset)
    columns = dataset.getColumns()
    values, protected_values = dataset.getValues(), protected_dataset.getValues()
    distance = 0
    for i in range(size):
        for j in range(len(columns)):
            # chol,location,age,gender,height,frame,waist
            if columns[j] == "chol":
                d = (values[i][j] - protected_values[i][j]) / 100
            elif columns[j] == "age":
                d = (values[i][j] - protected_values[i][j]) / 50
            elif columns[j] == "height":
                d = (values[i][j] - protected_values[i][j]) / 15
            elif columns[j] == "waist":
                d = (values[i][j] - protected_values[i][j]) / 20
            elif columns[j] == "frame":
                if values[i][j] == "medium" or protected_values[i][j] == "medium":
                    d = 1
                else:
                    d = 2
            else: #if columns[j] in ("location", "gender"):
                d = 1 if values[i][j] == protected_values[i][j] else 0
            
            distance += d
    return distance


SEP = "#############################################################"

def parse_k_anonimity(dataset: CSVReader, min: int = 0):
    columns = dataset.getColumns()
    for i in range(1, len(columns)+1):
        for combo in combinations(columns, i):
            indexes = [columns.index(col) for col in combo]
            k = k_anonymity(dataset, indexes)
            if k > min:
                print(f"Test combination : {combo}")
                print("k-anonimity :", k)

def generalize(reader: CSVReader, function, column: str):
    columns = reader.getColumns()
    index = columns.index(column)
    assert index >= 0, f"Column not found {column}"
    reader.map(lambda values: function(values, index), in_place=True)

def generalize_int(values: list, index: int):
    age: float = values[index]
    values[index] = 10 * (int(age) // 10)
    return values


In [362]:
directory = os.path.dirname(os.curdir)
diabetes1 = CSVReader.fromFile(os.path.join(directory, "diabetes1.csv"))
diabetes2 = CSVReader.fromFile(os.path.join(directory, "diabetes2.csv"))

protected_diabetes1 = diabetes1.copy()
protected_diabetes2 = diabetes2.copy()

## A

In [363]:
print(SEP)
print("Test K anonymity for diabetes1")
parse_k_anonimity(diabetes1, 2)

print(SEP)
print("Test K anonymity for diabetes2")
parse_k_anonimity(diabetes2, 2)

all_diabetes = diabetes1.union(diabetes2)
print(SEP)
print("Test K anonymity for both diabetes1 and diabetes2")
parse_k_anonimity(all_diabetes, 2)

#############################################################
Test K anonymity for diabetes1
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('location', 'gender')
k-anonimity : 81
#############################################################
Test K anonymity for diabetes2
Test combination : ('frame',)
k-anonimity : 100
#############################################################
Test K anonymity for both diabetes1 and diabetes2
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('location', 'gender')
k-anonimity : 81
Test combination : ('location', 'frame')
k-anonimity : 38
Test combination : ('gender', 'frame')
k-anonimity : 34
Test combination : ('location', 'gender', 'frame')
k-anonimity : 10


## B

In [364]:
generalize(protected_diabetes1, generalize_int, "chol")
generalize(protected_diabetes1, generalize_int, "age")

generalize(protected_diabetes2, generalize_int, "height")
generalize(protected_diabetes2, generalize_int, "waist")

print(SEP)
print("Test K anonymity for protected diabetes1")
parse_k_anonimity(protected_diabetes1, 2)

print(SEP)
print("Test K anonymity for protected diabetes2")
parse_k_anonimity(protected_diabetes2, 2)

protected_all_diabetes = protected_diabetes1.union(protected_diabetes2)
print(SEP)
print("Test K anonymity for both protected diabetes1 and protected diabetes2")
parse_k_anonimity(protected_all_diabetes, 2)

#############################################################
Test K anonymity for protected diabetes1
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('location', 'gender')
k-anonimity : 81
#############################################################
Test K anonymity for protected diabetes2
Test combination : ('height',)
k-anonimity : 14
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('waist',)
k-anonimity : 13
#############################################################
Test K anonymity for both protected diabetes1 and protected diabetes2
Test combination : ('location',)
k-anonimity : 188
Test combination : ('gender',)
k-anonimity : 162
Test combination : ('height',)
k-anonimity : 14
Test combination : ('frame',)
k-anonimity : 100
Test combination : ('waist',)
k-anonimity : 13
Test combination : ('location', 'gender')
k-anonimity : 81
Test combination : ('location', 'height')
k-anonimity : 4


## C

In [365]:
print("Distance for diabetes1:", distance(diabetes1, protected_diabetes1))
print("Distance for diabetes2:", distance(diabetes2, protected_diabetes2))
print("Distance for all diabetes:", distance(all_diabetes, protected_all_diabetes))

Distance for diabetes1: 817.619999999996
Distance for diabetes2: 784.7499999999995
Distance for all diabetes: 1602.3699999999928


Adding Differential Privacy would optimizes our suppressions and generalizations such that in general the adding or removing of a specific user doesn't change too much an output result (predefined could be anything : mean, variance, more complex things). For example, don't add a King Kong in an animal database his removal should be quite visible.

# Part 2

In [366]:
basket = CSVReader.fromFile("./basket.csv")
basket_test = CSVReader.fromFile("./basket_test.csv")
print(basket)

('milk', 'meat', 'apple', 'bread', 'pizza', 'beer', 'banana', 'fish', 'sugar', 'corn flakes', 'id')
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 0.0, 1.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0]
[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0]
[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0, 1.0, 1.0]
[1.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 1.0]
[1.0

In [367]:
class HypotheticalInformation:
    def __init__(self, reader: CSVReader, id_column: str):
        self.reader = reader
        assert id_column in reader.getColumns()
        self.id_column = id_column
        tmp = self.reader.getColumns()
        self.index = tmp.index(id_column)
        columns = []
        for each in tmp:
            if each != self.id_column:
                columns.append(each)
        self.columns = columns

    def __str_stats_list(self, references: list, lst: list):
        assert len(references) == len(lst)
        string = "["
        for i in range(len(lst)):
            string += "{" + f"{references[i]}:{round(lst[i], 3)}" + "}"
        return string + "]\n"

    def __str__(self):
        string = ""
        string += f"Proba per user: {self.getProbabilityPerUser()}\n"
        string += f"Shannon Entropy: {self.getShannonEntropy()}\n"
        string += f"Hypothetical Information: {self.getHypotheticalInformation()}\n"
        string += f"Perceived Information: {self.getPerceivedInformation(self.columns)}\n"
        
        ids = self.getUsers()
        nusers = len(ids)
        columns = self.columns

        ncolumns = len(columns)
        (P_U, P_O_IF_U), (P_O, P_U_IF_O) = self.getProbabilities()
        
        string += f"IDs: {ids}\n"
        string += f"Users stats :\n"
        for i in range(nusers):
            string += f"\tUser '{ids[i]}' (p={round(P_U[i], 3)}): {self.__str_stats_list(columns, P_O_IF_U[i])}\n"
        
        string += f"Columns: {columns}\n"
        string += f"Objects stats :\n"
        for j in range(ncolumns):
            string += f"\t{columns[j]} (p={round(P_O[j], 3)}): {self.__str_stats_list(ids, P_U_IF_O[j])}\n"

        return string

    def __preprocessRow(self, row: list):
        newRow = [0] * (len(row) - 1)
        k = 0
        for j in range(len(row)):
            if j != self.index:
                newRow[k] = row[j]
                k += 1
        return newRow

    def getUsers(self):
        id_column = self.id_column
        columns = self.reader.getColumns()
        index = self.index
        users = []
        for row in self.reader.getValues():
            try:
                id = int(row[index])
            except ValueError:
                id = row[index]
            if id not in users:
                users.append(id)
        return users

    def getProbabilityPerUser(self, empirical=False):
        index = self.index
        values = self.reader.getValues()
        nrows = len(values)
        users = {}
        if empirical:
            for row in values:
                user_id = row[index]
                if users.get(user_id, None) == None:
                    users[user_id] = 0
                users[user_id] += 1 / nrows
        else:
            for row in values:
                user_id = row[index]
                if users.get(user_id, None) == None:
                    users[user_id] = 1
            keys = list(users.keys())
            nusers = len(keys)
            for key in keys:
                users[key] /= nusers
        return users

    def getShannonEntropy(self, empirical = False):
        P = self.getProbabilityPerUser(empirical=empirical)
        entropy = 0.0
        for u in P.keys():
            entropy -= P[u] * np.log2(P[u])
        return entropy
    
    def getProbabilities(self):
        id_column = self.id_column
        columns = self.columns
        ncolumns = len(columns)

        index = self.index
        ids = self.getUsers()
        nusers = len(ids)

        rows = self.reader.getValues()
        nrows = len(rows)

        S_U = [0] * nusers
        S_O = [0] * ncolumns
        S_U_AND_O = [[0] * ncolumns for _ in range(nusers)]

        for row in rows:
            try:
                id = int(row[index])
            except ValueError:
                id = row[index]
            id_index = ids.index(id)
            S_U[id_index] += 1

            row = self.__preprocessRow(row)
            for j in range(ncolumns):
                assert type(row[j]) == float or type(row[j]) == int, f"Row {row} should contains only numerical values"
                S_O[j] += row[j]
                S_U_AND_O[id_index][j] += row[j]
            
        P_U = [0] * nusers
        P_O = [0] * ncolumns
        P_U_AND_O = [[0] * ncolumns for _ in range(nusers)]

        P_O_IF_U = []
        P_U_IF_O = []

        for i in range(nusers):
            P_U[i] = S_U[i] / nrows

        for j in range(ncolumns):
            P_O[j] = S_O[j] / nrows

        for i in range(nusers):
            for j in range(ncolumns):
                P_U_AND_O[i][j] = S_U_AND_O[i][j] / nrows

        for i in range(nusers):
            P_O_IF_Ui = []
            for j in range(ncolumns):
                P_O_IF_Ui.append(P_U_AND_O[i][j] / P_U[i])
            P_O_IF_U.append(P_O_IF_Ui)

        
        for j in range(ncolumns):
            P_U_IF_Oi = []
            for i in range(nusers):
                P_U_IF_Oi.append(P_U_AND_O[i][j] / P_O[j])
            P_U_IF_O.append(P_U_IF_Oi)


        return (P_U, P_O_IF_U), (P_O, P_U_IF_O)

    def getHypotheticalInformation(self, empirical = False):
        id_column = self.id_column
        columns = self.columns
        ncolumns = len(columns)

        index = self.index
        ids = self.getUsers()
        nusers = len(ids)

        H_U = self.getShannonEntropy(empirical=empirical)
        (P_U, P_O_IF_U), (P_O, P_U_IF_O) = self.getProbabilities()
        s = H_U
        for i in range(nusers):
            s2 = 0
            for j in range(ncolumns):
                s2 += P_O_IF_U[i][j] * np.log2(P_U_IF_O[j][i])
            s += P_U[i] * s2
        return s
    
    def getPerceivedInformation(self, test_columns: list, empirical = False):
        id_column = self.id_column
        columns = self.columns
        ncolumns = len(test_columns)

        index = self.index
        ids = self.getUsers()
        nusers = len(ids)

        H_U = self.getShannonEntropy(empirical=empirical)
        (P_U, P_O_IF_U), (P_O, P_U_IF_O) = self.getProbabilities()

        s = H_U
        for i in range(nusers):
            s2 = 0
            for j in range(ncolumns):
                s2 += np.log2(P_U_IF_O[j][i])
            s += P_U[i] * (s2 / len(test_columns))
        return s



## A, B and E

In [368]:
HI = HypotheticalInformation(basket, "id")
print(HI)


Proba per user: {1.0: 0.1, 2.0: 0.1, 3.0: 0.1, 4.0: 0.1, 5.0: 0.1, 6.0: 0.1, 7.0: 0.1, 8.0: 0.1, 9.0: 0.1, 10.0: 0.1}
Shannon Entropy: 3.321928094887362
Hypothetical Information: -13.873724385243168
Perceived Information: -0.05441733424927775
IDs: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Users stats :
	User '1' (p=0.1): [{milk:1.0}{meat:0.64}{apple:0.06}{bread:0.66}{pizza:0.6}{beer:0.36}{banana:0.34}{fish:0.36}{sugar:0.94}{corn flakes:0.64}]

	User '2' (p=0.1): [{milk:1.0}{meat:0.52}{apple:0.22}{bread:0.72}{pizza:0.5}{beer:0.46}{banana:0.28}{fish:0.46}{sugar:0.78}{corn flakes:0.52}]

	User '3' (p=0.1): [{milk:1.0}{meat:0.47}{apple:0.13}{bread:0.65}{pizza:0.52}{beer:0.25}{banana:0.35}{fish:0.25}{sugar:0.87}{corn flakes:0.47}]

	User '4' (p=0.1): [{milk:1.0}{meat:0.6}{apple:0.09}{bread:0.68}{pizza:0.59}{beer:0.37}{banana:0.32}{fish:0.37}{sugar:0.91}{corn flakes:0.6}]

	User '5' (p=0.1): [{milk:1.0}{meat:0.33}{apple:0.1}{bread:0.84}{pizza:0.74}{beer:0.27}{banana:0.16}{fish:0.27}{sugar:0.9}{corn fl

Shannon entropy measures uncertainty : the expected amount of information needed to identify the value of a random variable. The Shannon entropy represents an amount of the information, that is the number of questions needed **in average** to identify a specific user.

A low entropy is a bad sign, an attacker can identify very easily some users but an high one is a good sign as he needs more clues to identify someone.